In [ ]:
!pip -q install pandas
!pip -q install kagglehub
!pip -q install sentence-transformers
!pip -q install chromadb
!pip -q install transformers
!pip -q install accelerate
!pip -q install tqdm
!pip -q install streamlit
!pip -q install pyngrok

In [ ]:
import pandas as pd
import os
from tqdm import tqdm

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("Cornell-University/arxiv")

print("Path to dataset files:", path)

In [ ]:
import os

print(os.listdir(path))

In [ ]:
import glob

files = glob.glob(path + "/*")
files

In [ ]:
import pandas as pd
import os

json_file = os.path.join(path, "arxiv-metadata-oai-snapshot.json")

# Read only the first 5000 papers for development
df = pd.read_json(
    json_file,
    lines=True,
    nrows=5000
)

print(df.shape)
df.head()

In [ ]:
print(df.columns)

In [ ]:
cs_df = df[
    [
        "title",
        "abstract",
        "categories"
    ]
]

cs_df.head()

In [ ]:
cs_df = cs_df[
    cs_df["categories"].str.contains("cs", na=False)
]

print(cs_df.shape)

In [ ]:
cs_df = cs_df.dropna()

cs_df = cs_df.drop_duplicates()

cs_df.reset_index(drop=True, inplace=True)

print(cs_df.shape)

In [ ]:
cs_df.to_csv(
    "computer_science_papers.csv",
    index=False
)

print("Computer Science dataset saved successfully!")

In [ ]:
cs_df.head()

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded!")

In [ ]:
embeddings = embedding_model.encode(
    cs_df["abstract"].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Embeddings Shape:", embeddings.shape)

In [ ]:
import chromadb

client = chromadb.PersistentClient(
    path="arxiv_chroma_db"
)

collection = client.get_or_create_collection(
    "computer_science_papers"
)

print("ChromaDB Ready!")

In [ ]:
from tqdm import tqdm

for i in tqdm(range(len(cs_df))):

    title = str(cs_df.iloc[i]["title"])
    abstract = str(cs_df.iloc[i]["abstract"])
    category = str(cs_df.iloc[i]["categories"])

    collection.add(

        ids=[str(i)],

        embeddings=[embeddings[i].tolist()],

        documents=[abstract],

        metadatas=[
            {
                "title": title,
                "category": category
            }
        ]
    )

print("Database Created Successfully!")

In [ ]:
print(collection.count())

In [ ]:
def search_papers(query, n_results=3):

    query_embedding = embedding_model.encode(
        query
    ).tolist()

    results = collection.query(

        query_embeddings=[query_embedding],

        n_results=n_results

    )

    return results

In [ ]:
results = search_papers(
    "Deep Learning"
)

results

In [ ]:
def display_results(results):

    for i in range(len(results["documents"][0])):

        print("=" * 80)

        print("TITLE")

        print(results["metadatas"][0][i]["title"])

        print()

        print("CATEGORY")

        print(results["metadatas"][0][i]["category"])

        print()

        print("ABSTRACT")

        print(results["documents"][0][i])

        print()

In [ ]:
results = search_papers(
    "Computer Vision"
)

display_results(results)

In [ ]:
while True:

    query = input("\nSearch Paper (type exit to stop): ")

    if query.lower() == "exit":
        break

    results = search_papers(query)

    display_results(results)

In [ ]:
def retrieve_top_paper(question):

    result = search_papers(question, 1)

    paper = {

        "title": result["metadatas"][0][0]["title"],

        "category": result["metadatas"][0][0]["category"],

        "abstract": result["documents"][0][0]

    }

    return paper

In [ ]:
paper = retrieve_top_paper(
    "Large Language Models"
)

paper

In [ ]:
import shutil

shutil.make_archive(
    "arxiv_vector_database",
    "zip",
    "arxiv_chroma_db"
)

print("Database Saved Successfully!")

In [ ]:
!pip -q install transformers accelerate sentencepiece

In [ ]:
import transformers

print(transformers.__version__)

In [ ]:
from transformers import pipeline

llm = pipeline(
    "text-generation",
    model="google/flan-t5-base",
    max_new_tokens=256
)

print("Model Loaded Successfully!")

In [ ]:
!pip uninstall -y transformers

In [ ]:
!pip install -q transformers==4.53.2 sentencepiece accelerate

In [ ]:
import transformers

print(transformers.__version__)

In [ ]:
from transformers import pipeline
import torch

device = 0 if torch.cuda.is_available() else -1

llm = pipeline(
    task="text2text-generation",
    model="google/flan-t5-base",
    device=device
)

print("FLAN-T5 Loaded Successfully!")

In [ ]:
from transformers.pipelines import SUPPORTED_TASKS

print(sorted(SUPPORTED_TASKS.keys()))

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")

model = AutoModelForSeq2SeqLM.from_pretrained(
    "google/flan-t5-base"
).to(device)

print("FLAN-T5 Loaded Successfully!")

In [ ]:
prompt = "Explain Deep Learning in simple words."

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(device)

outputs = model.generate(
    **inputs,
    max_new_tokens=100
)

answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(answer)

In [ ]:
def summarize_paper(abstract):

    prompt = f"""
Summarize the following research paper in simple language.

Paper:
{abstract}
"""

    result = llm(prompt)

    return result[0]["generated_text"]

In [ ]:
print(search_papers("Deep Learning"))

In [ ]:
paper = retrieve_top_paper("Deep Learning")

summary = summarize_paper(paper["abstract"])

print(summary)

In [ ]:
def explain_concept(concept):

    prompt = f"""
Explain the following concept in simple words.

Concept:
{concept}
"""

    result = llm(prompt)

    return result[0]["generated_text"]

In [ ]:
print(explain_concept("Transformer Neural Networks"))

In [ ]:
def extract_information(abstract):

    prompt = f"""
Read the following research paper abstract and extract:

1. Main Problem
2. Proposed Method
3. Dataset Used
4. Results

Abstract:
{abstract}
"""

    result = llm(prompt)

    return result[0]["generated_text"]

In [ ]:
paper = retrieve_top_paper("Computer Vision")

print(extract_information(paper["abstract"]))

In [ ]:
def follow_up_question(abstract, question):

    prompt = f"""
Research Paper:

{abstract}

Question:

{question}
"""

    result = llm(prompt)

    return result[0]["generated_text"]

In [ ]:
paper = retrieve_top_paper("Large Language Models")

print(
    follow_up_question(
        paper["abstract"],
        "What problem does this paper solve?"
    )
)

In [ ]:
def expert_chatbot():

    print("=" * 70)
    print("ARXIV EXPERT CHATBOT")
    print("=" * 70)

    while True:

        query = input("\nEnter Research Topic (type 'exit' to quit): ")

        if query.lower() == "exit":
            break

        paper = retrieve_top_paper(query)

        print("\nTitle:")
        print(paper["title"])

        print("\nCategory:")
        print(paper["category"])

        print("\nSummary:")
        print(summarize_paper(paper["abstract"]))

        while True:

            follow = input("\nAsk a follow-up question (or type 'new'): ")

            if follow.lower() == "new":
                break

            print("\nAnswer:")
            print(follow_up_question(paper["abstract"], follow))

In [ ]:
expert_chatbot()

In [ ]:
!pip -q install streamlit pyngrok

In [ ]:
%%writefile app.py

import streamlit as st
import chromadb
from sentence_transformers import SentenceTransformer
from transformers import pipeline

# -----------------------------
# Load Models
# -----------------------------

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

llm = pipeline(
    "text2text-generation",
    model="google/flan-t5-base"
)

# -----------------------------
# Load ChromaDB
# -----------------------------

client = chromadb.PersistentClient(path="arxiv_chroma_db")

collection = client.get_collection("computer_science_papers")

# -----------------------------
# Search Function
# -----------------------------

def search_paper(query):

    embedding = embedding_model.encode(query).tolist()

    result = collection.query(
        query_embeddings=[embedding],
        n_results=1
    )

    paper = {
        "title": result["metadatas"][0][0]["title"],
        "category": result["metadatas"][0][0]["category"],
        "abstract": result["documents"][0][0]
    }

    return paper

# -----------------------------
# Summarization
# -----------------------------

def summarize(text):

    prompt = f"""
Summarize this research paper.

{text}
"""

    result = llm(prompt)

    return result[0]["generated_text"]

# -----------------------------
# Follow-up QA
# -----------------------------

def answer_question(abstract, question):

    prompt = f"""
Paper:

{abstract}

Question:

{question}
"""

    result = llm(prompt)

    return result[0]["generated_text"]

# -----------------------------
# Streamlit UI
# -----------------------------

st.set_page_config(page_title="Research Paper Chatbot")

st.title("📚 Expert Research Paper Chatbot")

topic = st.text_input("Enter Research Topic")

if st.button("Search"):

    paper = search_paper(topic)

    st.subheader("Title")

    st.write(paper["title"])

    st.subheader("Category")

    st.write(paper["category"])

    st.subheader("Summary")

    summary = summarize(paper["abstract"])

    st.success(summary)

    question = st.text_input(
        "Ask Follow-up Question"
    )

    if st.button("Answer"):

        ans = answer_question(
            paper["abstract"],
            question
        )

        st.info(ans)

In [ ]:
!streamlit run app.py & npx localtunnel --port 8501

In [ ]:
import matplotlib.pyplot as plt

top = cs_df["categories"].value_counts().head(10)

plt.figure(figsize=(10,5))

top.plot(kind="bar")

plt.title("Top Research Categories")

plt.show()

In [ ]:
!pip -q install wordcloud

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

text = " ".join(cs_df["abstract"])

cloud = WordCloud(
    width=1200,
    height=600,
    background_color="white"
).generate(text)

plt.figure(figsize=(15,7))

plt.imshow(cloud)

plt.axis("off")

plt.show()

In [ ]:
import shutil

shutil.make_archive(
    "research_database",
    "zip",
    "arxiv_chroma_db"
)

print("Database Saved!")